# PARADISEC archive

Extracts resource sizes and subject-language metadata from PARADISEC.

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from shapely.geometry import Point
import geopandas
import matplotlib.pyplot as plt
import json
import os

In [ ]:
paradisec = 'https://catalog.paradisec.org.au/collections/search'

In [ ]:
html_content = requests.get(paradisec).content

soup = BeautifulSoup(html_content, 'html.parser')

In [ ]:
a_tags = soup.find_all('a')

In [ ]:
paradisec_languages = [a_tag.text.strip() for a_tag in a_tags]

In [ ]:
len(paradisec_languages)

In [ ]:
glottolog_data = pd.read_csv('languages_and_dialects_geo.csv')

In [ ]:
df_filtered = glottolog_data[glottolog_data['name'].isin(paradisec_languages)]

In [ ]:
df_filtered

In [ ]:
points = df_filtered.apply(lambda r: Point(r.longitude, r.latitude), axis=1)
gdb = geopandas.GeoDataFrame(df_filtered.assign(geometry=points), \
                        crs={'init' :'epsg:4326'})
xys = gdb['geometry'].apply(lambda pt: pd.Series({'x': pt.x, 'y': pt.y}))
gdb['x'] = xys['x']
gdb['y'] = xys['y']

In [ ]:
worldmap = geopandas.read_file(geopandas.datasets.get_path("naturalearth_lowres"))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
worldmap.plot(color="white", edgecolor = 'black',ax=ax, linewidth=0.3)
gdb.plot(ax=ax, color = 'blue', markersize = 5)
plt.show()

# Size extraction

In [ ]:
graphql_query = """{
  items(limit: 100, page: 2) {
    total
    next_page
    results {
      full_identifier
      title
      description
      dialect
      language
      region
      originated_on
      essences_count
      essences {
        id   
        filename
        duration
        bitrate:bitrate
        samplerate
        itemId
        mimetype
        updatedAt
        fps
      }
      originated_on_narrative
      collection {
        identifier
        title
      }
      collector {
        first_name
        last_name
        country
      }
      university {
        name
        party_identifier
      }
      discourse_type {
        name
      }
      data_types {
        name
      }
      data_categories {
        name
      }
      countries {
        name
        code
      }
      subject_languages {
        name
        code
      }
      content_languages {
        name
        code
      }
      access_class
      access_narrative
      citation
      ingest_notes
      original_media
      permalink
      doi
    }
  }
}"""

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:124.0) Gecko/20100101 Firefox/124.0",
    "Accept": "*/*",
    "Accept-Language": "en-GB,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br",
    "Referer": "https://catalog.paradisec.org.au/graphiql",
    "Content-Type": "application/json",
    "X-CSRF-Token": "WMRdCmarai5DfoZrbM1MdTJtYh-7kx73DPKGr5cgNJn9R9JahgbhsScC4IG4UteTSgmpp-x2AtySU2w1pesaqA",
    "Origin": "https://catalog.paradisec.org.au",
    "Connection": "keep-alive",
    "Cookie": "_ga_X7RT9Y7NXS=GS1.1.1711475412.8.1.1711475953.0.0.0; _ga=GA1.3.1409180406.1711383017; _gid=GA1.3.1117965470.1711383018; _nabu_session=Na1v8PMml3%2BaGb4qjxRi%2F4WaI4jcFeTm5ocIlTaaHagMuZj6ht4mjqUHJ4IFdxyiT3yxy54umIUjL2b9P0UKThudJJEgSdtNXBH4SRcrepdPtCvWoCnSkmsKuk8v7gxcRIMyztcZgIyeKFUxtuMFEU20UIJJSqxucf%2FsFy5W2%2BXOoYHRhT8fCmbSabIyWkmcVkxJyldWcEMWGS4RSNXIAdV3KnEUjVztNjMuNSqFkDj0LJ17q0IDWsUEI7FKiuZ%2BRMobigJbxBr1h3i6Y6oY6Rst0GxzBB%2BAkAfkM2z2K%2FmKSFPql11%2Bk2xkpLwmOPU7yaJfypxMWTyO0EOrVz9UY2cEez90LduZxpB2QZuV7PZFIohwFmpG0FoMUvhpULJpPOn3GPGbaqQA1b23IKKHTFvPL2YAnsT%2Bf5JqP49rQTFyYidzLDTV6giLFLQzyPxJnsMvrv%2B2cqKRCHspqkCtYmNzLAsCCFksMFgLJYHKKkDv%2FGII3B9qFy0LuxDePyUa5WQNqkve6rz1l%2BskpXHOjr2Mj8KY7oROuvay61U139mcGznNEAA8scjdNH6WY%2Bi3Cf4V%2BNLR2lnWDJGObOwP2pU%2FR5W7KYUWK9UnacP7VJN9%2BMG%2FFCMIXjH9MWP2X87wYOatlGaaF2Y4lRzS6vTlFE8zY74LUH9Skwdl669INJfuZcvocxYJ7DA4F6mWyAkirEl%2F6aq%2F2b9E--QlwX8pBq%2FxmRznVV--Z4LJfJGbMFTJNgy%2F5Mjc8w%3D%3D",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
    "TE": "trailers"
}

In [ ]:
url = 'https://catalog.paradisec.org.au/graphiql'

In [ ]:

response = requests.post(url=url, json=graphql_query, headers=headers)

In [ ]:
response

In [ ]:
tr = response.json()

In [ ]:
def run_query(query, headers=None): # A simple function to use requests.post to make the API call. Note the json= section.
    if headers is None:
        headers = {}
    request = requests.post('https://catalog.paradisec.org.au/graphiql', json={'query': query}, headers=headers)
    if request.status_code == 200:
        return request.json()
    else:
        raise Exception("Query failed to run by returning code of {}. {}".format(request.status_code, query))

In [ ]:
request = requests.post('https://catalog.paradisec.org.au/graphiql', json={'query': graphql_query}, headers = headers)

In [ ]:
request

In [ ]:
result = run_query(graphql_query)

In [ ]:
url = 'https://catalog.paradisec.org.au/graphiql'  # Replace 'example.com/graphql' with your actual GraphQL endpoint

# Define the query
query = {
    "query": """
        {
          items(limit: 10000, page: 5) {
            total
            next_page
            results {
              full_identifier
              title
              description
              dialect
              language
              region
              originated_on
              essences_count
              essences {
                id
                filename
                duration
                bitrate: bitrate
                samplerate
                itemId
                mimetype
                updatedAt
                fps
              }
              originated_on_narrative
              collection {
                identifier
                title
              }
              collector {
                first_name
                last_name
                country
              }
              countries {
                name
                code
              }
              subject_languages {
                name
                code
              }
              content_languages {
                name
                code
              }
            }
          }
        }
    """
}

# Send the HTTP request
response = requests.post(url, json=query)

In [ ]:
response

In [ ]:
tr

In [ ]:
type(graphql_query)

In [ ]:
my_json = response.content.decode('utf8').replace("'", '"')

In [ ]:
my_json

# get subject languages name and code

In [ ]:
with open('paradisec.json', 'r') as file:
    # Load JSON data from the file
    data = json.load(file)

In [ ]:
data

In [ ]:
folder_path = 'data/paradisec_data'

In [ ]:
# Initialize empty lists to store data
rows = []


for filename in os.listdir(folder_path):
    if filename.startswith('paradisec') and filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'r') as file:
            # Load JSON data from the file
            data = json.load(file)
# Extract data from each result
        for result in data['data']['items']['results']:
            sizes = []
            subject_language_names = []
            subject_language_codes = []
            content_language_names = []
            content_language_codes = []
            
            # Extract size
            for essence in result['essences']:
                sizes.append(int(essence['size']))
            
            # Extract subject languages
            for subject_language in result['subject_languages']:
                subject_language_names.append(subject_language['name'])
                subject_language_codes.append(subject_language['code'])
            
            # Extract content languages
            for content_language in result['content_languages']:
                content_language_names.append(content_language['name'])
                content_language_codes.append(content_language['code'])

            # Add extracted data to rows list
            rows.append({
                'Size': sizes,
                'Subject Language Name': subject_language_names,
                'Subject Language Code': subject_language_codes,
                'Content Language Name': content_language_names,
                'Content Language Code': content_language_codes
            })

        # Create DataFrame
        df = pd.DataFrame(rows)

        # Display DataFrame
        print(df)

In [ ]:
df

In [ ]:
df['Subject Language Name'] = df['Subject Language Name'].apply(lambda x: ', '.join(x))

In [ ]:
df['Subject Language Code'] = df['Subject Language Code'].apply(lambda x: ', '.join(x))

In [ ]:
df['Size'] = df['Size'].apply(lambda x: sum(x))

In [ ]:
df

In [ ]:
summed_df = df.groupby(['Subject Language Name', 'Subject Language Code'])['Size'].sum().reset_index()

In [ ]:
summed_df = df.groupby(['Subject Language Name'])['Size'].sum().reset_index()

In [ ]:
summed_df

In [ ]:
summed_df['Size'] = summed_df['Size'] / (1024 ** 3)

In [ ]:
df_sorted = summed_df.sort_values(by='Size', ascending=False)

In [ ]:
df_sorted

In [ ]:
df_sorted.to_excel("paradisec.xlsx")

In [ ]:
Language = []
Size = []

In [ ]:
languages = []
sizes = []

for result in data['data']['items']['results']:
    # Check if 'essences' list is not empty
    if result['essences']:
        # Extract language and size from each result and append to the lists
        languages.append(result['language'])
        sizes.append(result['essences'][0]['size'])

In [ ]:
df = pd.DataFrame({'Language': languages, 'Size': sizes})

In [ ]:
df

In [ ]:
aggregated_df = df.groupby('Language').sum().reset_index()

In [ ]:
aggregated_df

In [ ]:
df

In [ ]:
aggregated_df.to_csv('paradisec.csv', index=False)